# CS164 Optimization Methods — Session 15: Applying & Interpreting the KKT Conditions

**Source:** Reconstructed from the submitted *CS164 Session 15 PCW.pdf* (six pages), rather than an original notebook export. The PCW's questions, flavor data, written reasoning, and R active-set code are preserved in organized form. Numerical answers were independently checked; the R cells have not been executed as part of this reconstruction.

**Topic:** KKT applications, nonnegative least squares, active constraints, and the interpretation of inequality multipliers.

## Question 1 — Review

The prior session introduced the Karush–Kuhn–Tucker (KKT) conditions for optimization with inequality constraints. This session applies those conditions to a concrete mixture-fitting problem and interprets the Lagrange multipliers, especially the multipliers $\mu$ attached to inequality constraints.

## Nonnegative Least Squares Fitting — Problem setup

Three drinks are scored across six flavor dimensions (sugar, acid, body, fruity, flowery, intensity):

| Drink | Sugar | Acid | Body | Fruity | Flowery | Intensity |
|---|---:|---:|---:|---:|---:|---:|
| 1 | 1 | 1 | 2 | 3 | 2 | 5 |
| 2 | 2 | 1 | 2 | 3 | 2 | 5 |
| 3 | 1 | 3 | 5 | 1 | 3 | 1 |

Their profiles form $X\in\mathbb{R}^{3\times6}$, and the target is $y=(3,2,2,2,3,2)^T$. The decision vector $\theta=(\theta_1,\theta_2,\theta_3)^T$ specifies the proportions of the drinks.

## Question 2 — Why constrain the proportions?

Each $\theta_i$ represents a physical fraction, so $\theta_i\geq0$; a zero proportion means the corresponding drink is excluded. The proportions must add up to the entire mixture, so $\theta_1+\theta_2+\theta_3=1$.

The feasible set is the **standard 2-simplex**: $\{\theta\in\mathbb R^3:\theta\geq0,\ \mathbf1^T\theta=1\}$.

## Question 3 — Least-squares optimization problem

$X^T\theta\in\mathbb R^6$ is the mixture's predicted flavor profile. To match $y$ in the least-squares sense, solve

$$
\begin{aligned}
\min_{\theta\in\mathbb R^3}\quad & \|X^T\theta-y\|_2^2\\
\text{subject to}\quad & \theta_i\geq0\quad(i=1,2,3),\\
& \mathbf1^T\theta=1.
\end{aligned}
$$

The subscript $2$ denotes the Euclidean norm. Squaring the norm turns the magnitude of the residual vector into the **sum of squared residuals**. Because squaring is increasing on nonnegative numbers, the squared and unsquared Euclidean norms have the same minimizers. The squared form is more convenient for gradients and gives a convex quadratic objective:

$$
\|X^T\theta-y\|_2^2
=\theta^TXX^T\theta-2y^TX^T\theta+y^Ty.
$$

## Question 4 — Lagrangian and KKT conditions

Use $\lambda\in\mathbb R$ for the equality and $\mu\in\mathbb R_{\geq0}^3$ for the inequalities $-\theta_i\leq0$:

$$
\mathcal L(\theta,\lambda,\mu)
=\|X^T\theta-y\|_2^2
+\lambda(\mathbf1^T\theta-1)-\mu^T\theta.
$$

The four KKT conditions are:

1. **Stationarity:** $2X(X^T\theta-y)+\lambda\mathbf1-\mu=0$.
2. **Primal feasibility:** $\theta\geq0$ and $\mathbf1^T\theta=1$.
3. **Dual feasibility:** $\mu\geq0$.
4. **Complementary slackness:** $\mu_i\theta_i=0$ for $i=1,2,3$.

The objective is convex quadratic, the constraints are affine, and a strictly positive feasible point $\theta=(1/3,1/3,1/3)$ exists. Thus the KKT conditions characterize global optimality.

## Question 5 — Solve the KKT system with eight active sets

Complementary slackness implies, for every $i$, either $\theta_i=0$ (the nonnegativity constraint is active) or $\mu_i=0$. There are $2^3=8$ combinations to examine.

Below is the R implementation from the submitted PCW, with minor formatting and naming cleanup. For each active set, it solves the stationarity equations and $\mathbf1^T\theta=1$, then checks whether both $\theta$ and $\mu$ are nonnegative.

In [ ]:
# Session 15 PCW: enumerate all eight active sets in R
X <- rbind(
  c(1, 1, 2, 3, 2, 5),
  c(2, 1, 2, 3, 2, 5),
  c(1, 3, 5, 1, 3, 1)
)

y <- c(3, 2, 2, 2, 3, 2)

# Stationarity: H %*% theta - b + lambda - mu = 0
H <- 2 * X %*% t(X)
b <- as.vector(2 * X %*% y)

# All 8 combinations of active nonnegativity constraints
cases <- expand.grid(
  a1 = c(FALSE, TRUE),
  a2 = c(FALSE, TRUE),
  a3 = c(FALSE, TRUE)
)

for (j in seq_len(nrow(cases))) {
  active <- which(unlist(cases[j, ]))
  free <- setdiff(1:3, active)

  label <- if (length(active) == 0) "none" else
    paste(active, collapse = ",")

  if (length(free) == 0) {
    cat("Active:", label, "- Infeasible (sum cannot equal 1)\n")
    next
  }

  # Solve stationarity for free proportions, plus sum(theta) = 1
  k <- length(free)
  K <- rbind(
    cbind(H[free, free, drop = FALSE], rep(1, k)),
    c(rep(1, k), 0)
  )

  solution <- solve(K, c(b[free], 1))

  theta <- numeric(3)
  theta[free] <- solution[1:k]
  lambda <- solution[k + 1]

  mu <- as.vector(H %*% theta - b + lambda)
  mu[free] <- 0

  squared_error <- sum((as.vector(t(X) %*% theta) - y)^2)
  valid <- all(theta >= -1e-9) && all(mu >= -1e-9)

  cat("\nActive:", label,
      "\ntheta:", round(theta, 6),
      "\nmu:", round(mu, 6),
      "\nlambda:", round(lambda, 6),
      "\nSquared error:", round(squared_error, 6),
      "\nKKT satisfied:", valid, "\n")

  if (valid) {
    theta_opt <- theta
    lambda_opt <- lambda
    mu_opt <- mu
  }
}

cat("\nOptimal mixture:", round(theta_opt, 6), "\n")

### Results and optimum

Exactly one active-set candidate satisfies all KKT conditions: **only $\theta_1=0$ is active**. The valid solution is

$$
\boxed{\theta^*=\left(0,\frac{19}{35},\frac{16}{35}\right)
\approx(0,\ 0.542857,\ 0.457143).}
$$

The multipliers and minimum squared residual are

$$
\lambda^*=-\frac{334}{35}\approx-9.542857,\qquad
\mu^*=\left(\frac{102}{35},0,0\right)\approx(2.914286,0,0),
$$

$$
\|X^T\theta^*-y\|_2^2=\frac{199}{35}\approx5.685714.
$$

**Interpretation:** the closest physically possible blend is **0% Drink 1, 54.29% Drink 2, and 45.71% Drink 3**. Some other active-set solutions have nonnegative proportions but violate dual feasibility; others have negative proportions. The case in which all three proportions are zero is impossible because they must sum to one.

## Question 6 — Meaning of nonzero inequality multipliers

In the optimal solution, $\mu_1>0$ and $\mu_2=\mu_3=0$. The positive multiplier means the **Drink 1 nonnegativity restriction is binding and consequential**: allowing a slightly negative $\theta_1$ would decrease the squared error. A multiplier's size is a local sensitivity (shadow price) associated with relaxing its constraint.

If we remove only the nonnegativity conditions, **while keeping** $\mathbf1^T\theta=1$, the best-fitting affine combination becomes

$$
\theta_{\mathrm{affine}}=\left(-\frac32,\ 2,\ \frac12\right),
\qquad \|X^T\theta_{\mathrm{affine}}-y\|_2^2=3.5.
$$

This fit is better numerically than $199/35$, but its negative Drink 1 fraction is physically impossible. Thus the admissible optimum is pushed onto the simplex boundary $\theta_1=0$.

*Terminology:* The PCW refers to the “true line of best fit”; here, this comparison is between the simplex-constrained blend and the equality-constrained least-squares affine fit.

### Optional R check — equality-constrained fit without nonnegativity

This extends the PCW by calculating the affine fit directly (the equality constraint still applies).

In [ ]:
# KKT equations with no nonnegativity multipliers
K_affine <- rbind(
  cbind(H, rep(1, 3)),
  c(rep(1, 3), 0)
)
theta_affine <- as.vector(solve(K_affine, c(b, 1))[1:3])
error_affine <- sum((as.vector(t(X) %*% theta_affine) - y)^2)

cat("Equality-only solution:", round(theta_affine, 6), "\n")
cat("Squared error:", round(error_affine, 6), "\n")

## Question 7 — Optional attachment

The PCW offered an optional handwritten-work upload. No additional handwritten attachment is included in the submitted six-page PDF.

**Source and execution note:** All seven question prompts and the student's R method were reconstructed from the supplied PDF. Any R code cell here requires an **IRkernel**-enabled Jupyter environment. The algebraic results were checked independently of R.